In [1]:
using BayesFlux, Flux
using Random, Distributions
using StatsPlots, Optim
using ARCHModels, LinearAlgebra, DataFrames, CSV, Plots, Statistics
using MCMCChains, Bijectors

# Load data
eft_rf = "/Users/kevin/Documents/University of Maastricht /Master Econometrics and Operations Research/Master Thesis /MasterThesis/Continuation of the BayesFlux/data/etfReturns.csv"
df = CSV.read(eft_rf, DataFrame)

etf_names = ["16383", "16386", "16388", "16397", "16403", "16412", "16414", "16418", 
             "16421", "16423", "16424", "16426", "16433", "16437", "16452", "16460", 
             "24697", "27635", "28272", "28273", "28274", "28275", "28276", "28277", 
             "28278", "28279", "28280", "31372", "31466"]



# Data preprocessing
function preprocess_data(df, etf_names)
    # Handle missing values , the missing value is replaced by the mean of the column. "Ask the professor if it is the best way to handle missing values"
    for col in etf_names
        if any(ismissing, df[!, col])
            df[!, col] = coalesce.(df[!, col], mean(skipmissing(df[!, col])))
        end
    end
    
    etf_returns = Matrix{Float64}(df[!, etf_names])
    rf_returns = Vector{Float64}(df[!, "rf"])
    returns = hcat(etf_returns, rf_returns)
    
    return etf_returns, rf_returns, returns
end

# Process data
etf_returns, rf_returns, returns = preprocess_data(df, etf_names)

# GARCH(1,1) likelihood for parameter estimation
function garch_likelihood(params, returns)
    ω, α, β = params
    n = length(returns)
    σ² = zeros(n)
    σ²[1] = var(returns)
    loglik = 0.0
    
    for t in 2:n
        σ²[t] = max(ω + α * returns[t-1]^2 + β * σ²[t-1], 1e-6)
        loglik += -0.5 * (log(2π) + log(σ²[t]) + returns[t]^2/σ²[t])
    end
    
    return -loglik
end

# Estimate GARCH parameters
function estimate_garch_params(returns)
    initial_params = [var(returns)*0.01, 0.1, 0.8]
    
    function obj(params)
        ω, α, β = params
        if ω ≤ 0 || α < 0 || β < 0 || α + β ≥ 1
            return Inf
        end
        return garch_likelihood(params, returns)
    end
    
    result = optimize(obj, initial_params, BFGS())
    return Optim.minimizer(result)
end

# Fit univariate GARCH(1,1)
function fit_univariate_garch(returns)
    n = length(returns)
    ω, α, β = estimate_garch_params(returns)
    σ² = zeros(n + 1)
    σ²[1] = var(returns)
    
    for t in 2:n
        σ²[t] = ω + α * returns[t-1]^2 + β * σ²[t-1]
    end
    
    σ²[n+1] = ω + α * returns[n]^2 + β * σ²[n]
    return σ², (ω, α, β)
end

# DCC likelihood for parameter estimation
function dcc_likelihood(params, std_returns, Q_bar)
    a, b = params
    T, N = size(std_returns)
    Qt = similar(Q_bar)
    Qt .= Q_bar
    loglik = 0.0
    
    for t in 2:T
        Qt = (1 - a - b) * Q_bar + 
             a * (std_returns[t-1,:] * std_returns[t-1,:]') + 
             b * Qt
        
        Qt_diag = Diagonal(sqrt.(diag(Qt)))
        Rt = inv(Qt_diag) * Qt * inv(Qt_diag)
        
        loglik += -0.5 * (log(det(Rt)) + std_returns[t,:]' * inv(Rt) * std_returns[t,:])
    end
    
    return -loglik
end

# Estimate DCC parameters
function estimate_dcc_params(std_returns)
    Q_bar = cor(std_returns)
    
    function obj(params)
        a, b = params
        if a < 0 || b < 0 || (a + b) ≥ 1
            return Inf
        end
        return dcc_likelihood(params, std_returns, Q_bar)
    end
    
    initial_params = [0.01, 0.97]
    result = optimize(obj, initial_params, BFGS())
    return Optim.minimizer(result)
end

# Main DCC-GARCH function
function dcc_garch(returns)
    n, N = size(returns)
    
    # First stage: Fit univariate GARCH models
    volatilities = zeros(n + 1, N)
    std_returns = zeros(n, N)
    garch_params = Vector{Tuple{Float64, Float64, Float64}}(undef, N)
    
    for i in 1:N
        volatilities[:, i], garch_params[i] = fit_univariate_garch(returns[:, i])
        std_returns[:, i] = returns[:, i] ./ sqrt.(volatilities[1:end-1, i])
    end
    
    # Second stage: DCC estimation
    a, b = estimate_dcc_params(std_returns)
    
    Q_bar = cor(returns)
    Q_t = zeros(n, N, N)
    R_t = zeros(n, N, N)
    H_t = zeros(n, N, N)
    Q_t[1, :, :] = Q_bar
    
    for t in 2:n
        Q_t[t, :, :] = (1 - a - b) * Q_bar + 
                       a * (std_returns[t-1, :] * std_returns[t-1, :]') + 
                       b * Q_t[t-1, :, :]
        
        # Ensure positive definiteness
        Q_t[t, :, :] = (Q_t[t, :, :] + Q_t[t, :, :]') / 2
        
        # Compute correlation matrix
        Q_diag = Diagonal(sqrt.(diag(Q_t[t, :, :])))
        R_t[t, :, :] = inv(Q_diag) * Q_t[t, :, :] * inv(Q_diag)
        
        # Compute conditional covariance matrix
        D_t = Diagonal(sqrt.(volatilities[t, :]))
        H_t[t, :, :] = D_t * R_t[t, :, :] * D_t
    end
    
    return Dict(
        "correlations" => R_t,
        "volatilities" => volatilities,
        "std_returns" => std_returns,
        "garch_params" => garch_params,
        "dcc_params" => (a, b),
        "conditional_cov" => H_t
    )
end

# Analysis function
function analyse_results(dcc_results, etf_names, returns)
    n, N = size(returns)
    
    println("\nGARCH(1,1) Parameters for each asset:")
    for (i, etf) in enumerate(etf_names)
        ω, α, β = dcc_results["garch_params"][i]
        println("ETF $etf: ω = $ω, α = $α, β = $β")
    end
    
    a, b = dcc_results["dcc_params"]
    println("\nDCC Parameters:")
    println("a = $a")
    println("b = $b")
    
    println("\nAverage correlations with risk-free rate:")
    for (i, etf) in enumerate(etf_names)
        avg_corr = mean(dcc_results["correlations"][:, i, end])
        println("ETF $etf: $avg_corr")
    end
end



# Fit model
dcc_results = dcc_garch(returns)

analyse_results(dcc_results, etf_names, returns)

LoadError: MethodError: no method matching pipe_writer(::IJulia.IJuliaStdio{Base.PipeEndpoint})
The applicable method may be too new: running in world age 31486, while current world is 31500.

[0mClosest candidates are:
[0m  pipe_writer(::IJulia.IJuliaStdio) (method too new to be called from this world context.)
[0m[90m   @[39m [36mIJulia[39m [90m~/.julia/packages/IJulia/XF6bn/src/[39m[90m[4mstdio.jl:16[24m[39m
[0m  pipe_writer([91m::Base.ProcessChain[39m)
[0m[90m   @[39m [90mBase[39m [90m[4mprocess.jl:36[24m[39m
[0m  pipe_writer([91m::Base.Process[39m)
[0m[90m   @[39m [90mBase[39m [90m[4mprocess.jl:22[24m[39m
[0m  ...
